# CycleGAN Hand OA Progression Synthesis — Workflow, Analysis & Results Notebook

This notebook provides an end-to-end visualization and verification suite for the **CycleGAN (`KL0 -> KL3` on DIP2–DIP5)** pipeline on BU SCC:

1. **Dataset & Anti-Leakage Splits**: Patient-level 3-way partition (`train` / `val` / `test`), severe KL-grade imbalance, and joint breakdown.
2. **CycleGAN Optimization Curves**: 90-epoch Generator, Cycle-Consistency, Adversarial, and PatchGAN Discriminator loss trajectories.
3. **Validation Metrics & Early Stopping**: `val_kid`, `val_fid`, `val_cycle_ssim`, `val_struct_ssim`, and `val_delta_l1` across checkpoints, justifying **Epoch 50** selection.
4. **Qualitative Visual Inspection**: Annotated validation cycle grids (`Real -> Fake -> Reconstructed`), cross-epoch progression (`Epoch 5 -> 30 -> 50 -> 90`), and `Synthetic KL3 - Real KL0` difference heatmaps.
5. **Downstream VGG19 Evaluation (`test` set)**: Four-strategy benchmark (`Baseline CE`, `CycleGAN SD + CE`, `OS + Focal`, `CycleGAN SD + OS + Focal`), ablation deltas, and confusion matrices.
6. **Full CycleGAN Training (Optional Execution)**: Full YAML-aligned training cell (`200` epochs, AMP `bfloat16`, early stopping, checkpoint resume; guarded by `RUN_TRAINING = False`).
7. **Full Synthetic KL3 Generation (Optional Execution)**: Full pool generation cell (`300` train `KL0 -> KL3` images from `best_checkpoint.pt`; guarded by `RUN_SYNTHESIS = False`) plus BU SCC `.qsub` batch commands.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "configs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

for path_entry in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")):
    if path_entry not in sys.path:
        sys.path.insert(0, path_entry)

from hand_oa.data import (
    load_manifest_from_config,
    summarize_manifest,
    verify_no_data_leakage,
)
from hand_oa.utils import load_yaml_config, resolve_project_path

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.28,
    "grid.linestyle": "--",
    "font.size": 10,
})

data_cfg = load_yaml_config(resolve_project_path("configs/data.yaml", PROJECT_ROOT))
cyclegan_cfg = load_yaml_config(resolve_project_path("configs/cyclegan.yaml", PROJECT_ROOT))
cls_cfg = load_yaml_config(resolve_project_path("configs/classifier.yaml", PROJECT_ROOT))

cg_out_dir = resolve_project_path(cyclegan_cfg["experiment"]["output_dir"], PROJECT_ROOT)
eval_out_dir = resolve_project_path(cls_cfg["experiment"]["output_dir"], PROJECT_ROOT)
syn_dir = cg_out_dir / str(cyclegan_cfg["generation"]["output_subdir"])

artifacts = {
    "manifest.csv": resolve_project_path(data_cfg["paths"]["manifest_path"], PROJECT_ROOT),
    "loss_history.csv": cg_out_dir / "loss_history.csv",
    "val_metrics.csv": cg_out_dir / "val_metrics.csv",
    "best_checkpoint_info.json": cg_out_dir / "best_checkpoint_info.json",
    "best_checkpoint.pt": cg_out_dir / "checkpoints" / "best_checkpoint.pt",
    "synthetic_manifest.csv": syn_dir / "synthetic_manifest.csv",
    "eval_baseline_ce": eval_out_dir / "baseline_ce_test_metrics.json",
    "eval_cyclegan_sd_ce": eval_out_dir / "cyclegan_sd_ce_test_metrics.json",
    "eval_baseline_os_focal": eval_out_dir / "baseline_os_focal_test_metrics.json",
    "eval_cyclegan_sd_os_focal": eval_out_dir / "cyclegan_sd_os_focal_test_metrics.json",
}

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")
status_df = pd.DataFrame([
    {"artifact": name, "exists": path.is_file(), "path": str(path)}
    for name, path in artifacts.items()
])
display(status_df)


## 1. Dataset Splits, Class Imbalance & Anti-Leakage Verification
Verifies patient-level isolation across `train` (`76.5%`), `val` (`8.5%`), and locked `test` (`15.0%`), and visualizes the severe minority shortage at **KL3** and **KL4**.

In [ ]:
manifest, images_dir = load_manifest_from_config(data_cfg, PROJECT_ROOT, overwrite=False)
verify_no_data_leakage(manifest)
split_summary = summarize_manifest(manifest)

train_pids = set(manifest.loc[manifest["split"] == "train", "patient_id"].astype(str))
val_pids = set(manifest.loc[manifest["split"] == "val", "patient_id"].astype(str))
test_pids = set(manifest.loc[manifest["split"] == "test", "patient_id"].astype(str))
assert len(train_pids & val_pids) == 0
assert len(train_pids & test_pids) == 0
assert len(val_pids & test_pids) == 0
print(f"Verified zero patient overlap | Total: {split_summary['total_patients']:,} patients, {split_summary['total_samples']:,} images")

rows = []
for split_name in ("train", "val", "test"):
    info = split_summary["splits"][split_name]
    dist = info["kl_distribution"]
    oa_cnt = int(dist["3"]) + int(dist["4"])
    rows.append({
        "Split": split_name.upper(),
        "Patients": info["patients"],
        "Images": info["samples"],
        "KL0": dist["0"],
        "KL1": dist["1"],
        "KL2": dist["2"],
        "KL3": dist["3"],
        "KL4": dist["4"],
        "OA (KL>=3) %": round(100.0 * oa_cnt / info["samples"], 2),
    })

total_dist = {str(g): int((manifest["kl_grade"] == g).sum()) for g in range(5)}
rows.append({
    "Split": "TOTAL",
    "Patients": split_summary["total_patients"],
    "Images": split_summary["total_samples"],
    "KL0": total_dist["0"],
    "KL1": total_dist["1"],
    "KL2": total_dist["2"],
    "KL3": total_dist["3"],
    "KL4": total_dist["4"],
    "OA (KL>=3) %": round(100.0 * (total_dist["3"] + total_dist["4"]) / split_summary["total_samples"], 2),
})
display(pd.DataFrame(rows))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.2), dpi=140)

kl_grades = [0, 1, 2, 3, 4]
kl_labels = [f"KL{g}" for g in kl_grades]
x = np.arange(len(kl_grades))
width = 0.26
split_styles = [
    ("train", "Train (76.5%)", "#059669"),
    ("val", "Val (8.5%)", "#D97706"),
    ("test", "Locked Test (15.0%)", "#DC2626"),
]

for idx, (sk, label, color) in enumerate(split_styles):
    counts = [split_summary["splits"][sk]["kl_distribution"][str(g)] for g in kl_grades]
    bars = ax1.bar(x + (idx - 1) * width, counts, width=width, label=label, color=color, alpha=0.9)
    for bar, cnt in zip(bars, counts):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.12,
            f"{cnt:,}",
            ha="center",
            va="bottom",
            fontsize=7.8,
            rotation=0,
        )

ax1.axvspan(2.55, 4.45, color="#FEF2F2", alpha=0.75, zorder=0)
ax1.text(3.5, 450, "Binary OA (KL >= 3)\nSevere Minority", ha="center", va="bottom", fontsize=8.4, fontweight="bold", color="#991B1B")
ax1.annotate("CycleGAN Source (Domain A)", xy=(0 - width, 6535), xytext=(0.45, 9500),
             arrowprops=dict(arrowstyle="->", color="#1E293B", lw=1.1), fontsize=8.2, fontweight="bold", color="#1E293B")
ax1.annotate("CycleGAN Target (Domain B)", xy=(3 - width, 163), xytext=(1.35, 1150),
             arrowprops=dict(arrowstyle="->", color="#1D4ED8", lw=1.1), fontsize=8.2, fontweight="bold", color="#1D4ED8")

ax1.set_yscale("log")
ax1.set_ylim(10, 26000)
ax1.set_xticks(x)
ax1.set_xticklabels(kl_labels, fontweight="bold")
ax1.set_ylabel("Image Count (log scale)")
ax1.set_title("KL-Grade Distribution Across Patient-Isolated Splits", fontweight="bold", pad=12)
ax1.legend(frameon=False, loc="upper right", fontsize=8.4)

joint_order = ["DIP2", "DIP3", "DIP4", "DIP5"]
joint_ct = pd.crosstab(manifest["joint"], manifest["kl_grade"]).reindex(index=joint_order, columns=kl_grades, fill_value=0)
joint_colors = ["#1E3A8A", "#2563EB", "#60A5FA", "#F59E0B", "#DC2626"]
bottom = np.zeros(len(joint_order))
for g, col in zip(kl_grades, joint_colors):
    vals = joint_ct[g].to_numpy()
    ax2.bar(joint_order, vals, bottom=bottom, label=f"KL{g}", color=col, alpha=0.9)
    bottom += vals

for idx, jname in enumerate(joint_order):
    kl3_cnt = int(joint_ct.loc[jname, 3])
    kl0_cnt = int(joint_ct.loc[jname, 0])
    tot_j = int(joint_ct.loc[jname].sum())
    ax2.text(idx, tot_j + 60, f"n={tot_j:,}\nKL3={kl3_cnt} ({kl0_cnt}:{kl3_cnt})", ha="center", va="bottom", fontsize=7.9, color="#1E293B")

ax2.set_ylim(0, joint_ct.sum(axis=1).max() * 1.25)
ax2.set_ylabel("Image Count")
ax2.set_title("DIP Joint Breakdown & KL0:KL3 Imbalance Ratio", fontweight="bold", pad=22)
ax2.legend(frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, 0.99), fontsize=8.2)

plt.tight_layout()
plt.show()


## 2. CycleGAN Training Dynamics (`loss_history.csv`)
Visualizes the 90-epoch optimization run (2x GPU DDP, AMP `bfloat16`, early-stopped at Epoch 90) across Generator/Cycle losses, Adversarial losses, and PatchGAN Discriminator losses (LSGAN theoretical Nash equilibrium = `0.25`).

In [ ]:
df_loss = pd.read_csv(artifacts["loss_history.csv"])
best_info = json.loads(artifacts["best_checkpoint_info.json"].read_text()) if artifacts["best_checkpoint_info.json"].is_file() else {}
best_ep = int(best_info.get("best_epoch", 50))

milestones = [1, 25, best_ep, int(df_loss["epoch"].max())]
milestones = sorted(set(ep for ep in milestones if ep in set(df_loss["epoch"])))
milestone_df = df_loss[df_loss["epoch"].isin(milestones)][
    ["epoch", "lr", "loss_g", "loss_cycle_a", "loss_cycle_b", "loss_gan_a2b", "loss_gan_b2a", "loss_d_a", "loss_d_b"]
].round(4)
print(f"Training ran epochs 1 -> {int(df_loss['epoch'].max())} | Best checkpoint epoch = {best_ep}")
display(milestone_df)

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.6), dpi=140)

# Panel A: Generator & Cycle Consistency
ax = axes[0]
ax.plot(df_loss["epoch"], df_loss["loss_g"], label="Generator Total (G)", color="#0F172A", lw=2.0)
ax.plot(df_loss["epoch"], df_loss["loss_cycle_a"], label="Cycle A (KL0 -> KL3 -> KL0)", color="#2563EB", lw=1.8)
ax.plot(df_loss["epoch"], df_loss["loss_cycle_b"], label="Cycle B (KL3 -> KL0 -> KL3)", color="#059669", lw=1.8)
ax.axvline(best_ep, color="#DC2626", ls="--", lw=1.4, label=f"Best Ckpt (Ep {best_ep})")
ax.axvspan(best_ep, df_loss["epoch"].max(), color="#FEF2F2", alpha=0.45)
ax.set_title("Generator & Cycle-Consistency Losses", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend(frameon=False, fontsize=8.2, loc="upper right")

# Panel B: Generator Adversarial Losses
ax = axes[1]
ax.plot(df_loss["epoch"], df_loss["loss_gan_a2b"], label="GAN A->B (KL0 -> KL3)", color="#7C3AED", lw=1.8)
ax.plot(df_loss["epoch"], df_loss["loss_gan_b2a"], label="GAN B->A (KL3 -> KL0)", color="#D97706", lw=1.8)
ax.axvline(best_ep, color="#DC2626", ls="--", lw=1.4, label=f"Best Ckpt (Ep {best_ep})")
ax.set_title("Generator Adversarial Losses (LSGAN)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Adversarial Loss")
ax.legend(frameon=False, fontsize=8.2, loc="upper right")

# Panel C: PatchGAN Discriminators
ax = axes[2]
ax.plot(df_loss["epoch"], df_loss["loss_d_a"], label="D_A (Real vs Fake KL0)", color="#0284C7", lw=1.8)
ax.plot(df_loss["epoch"], df_loss["loss_d_b"], label="D_B (Real vs Fake KL3)", color="#DB2777", lw=1.8)
ax.axhline(0.25, color="#475569", ls=":", lw=1.5, label="LSGAN Nash Eq. (0.25)")
ax.axvline(best_ep, color="#DC2626", ls="--", lw=1.4, label=f"Best Ckpt (Ep {best_ep})")
ax.set_title("PatchGAN Discriminator Balance", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("0.5 * (L_real + L_fake)")
ax.legend(frameon=False, fontsize=8.2, loc="upper right")

plt.tight_layout()
plt.show()


## 3. Validation Generative Metrics & Checkpoint Selection (`val_metrics.csv`)
Evaluates saved checkpoints every 5 epochs on the pure `val` split (`718` KL0 source images vs `23` KL3 target references). Checkpoint selection picks the lowest `val_kid` among epochs satisfying `val_cycle_ssim >= 0.75` and `val_delta_l1 >= 0.015`.

In [ ]:
df_val = pd.read_csv(artifacts["val_metrics.csv"])
es_cfg = cyclegan_cfg.get("early_stopping", {})
min_cycle_ssim = float(es_cfg.get("min_cycle_ssim", 0.75))
min_delta_l1 = float(es_cfg.get("min_delta_l1", 0.015))
patience_checks = int(es_cfg.get("patience", 8))

display_cols = [
    "epoch", "val_kid", "val_fid", "val_cycle_ssim", "val_struct_ssim", "val_cycle_l1", "val_delta_l1", "qualified"
]
display(df_val[display_cols].round(4))

best_row = df_val.loc[df_val["epoch"] == best_ep].iloc[0]
max_val_ep = int(df_val["epoch"].max())

fig, axes = plt.subplots(2, 2, figsize=(13.8, 8.2), dpi=140)

# 1. val_kid
ax = axes[0, 0]
ax.plot(df_val["epoch"], df_val["val_kid"], marker="o", lw=2.0, color="#2563EB", label="val_kid")
ax.scatter([best_ep], [best_row["val_kid"]], color="#DC2626", s=160, marker="*", zorder=5,
           label=f"Selected Best: Ep {best_ep} ({best_row['val_kid']:.4f})")
ax.axvspan(best_ep, max_val_ep, color="#FEF2F2", alpha=0.5,
           label=f"Early-Stop Patience ({patience_checks} checks = {max_val_ep - best_ep} ep)")
ax.annotate(f"Min KID = {best_row['val_kid']:.4f}\n(Epoch {best_ep})",
            xy=(best_ep, best_row["val_kid"]), xytext=(best_ep + 6, best_row["val_kid"] + 0.0045),
            arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.2), fontsize=8.8, fontweight="bold", color="#991B1B")
ax.set_title("Kernel Inception Distance (val_kid, Lower is Better)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("KID")
ax.legend(frameon=False, fontsize=8.2, loc="upper right")

# 2. val_fid
ax = axes[0, 1]
ax.plot(df_val["epoch"], df_val["val_fid"], marker="o", lw=2.0, color="#7C3AED", label="val_fid")
ax.scatter([best_ep], [best_row["val_fid"]], color="#DC2626", s=160, marker="*", zorder=5,
           label=f"Best Ckpt Ep {best_ep} ({best_row['val_fid']:.2f})")
ax.annotate(f"Min FID = {best_row['val_fid']:.2f}\n(Epoch {best_ep})",
            xy=(best_ep, best_row["val_fid"]), xytext=(best_ep + 6, best_row["val_fid"] + 4.0),
            arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.2), fontsize=8.8, fontweight="bold", color="#991B1B")
ax.set_title("Frechet Inception Distance (val_fid, Lower is Better)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("FID")
ax.legend(frameon=False, fontsize=8.2, loc="lower right")

# 3. Anatomical Preservation (SSIM)
ax = axes[1, 0]
ax.plot(df_val["epoch"], df_val["val_cycle_ssim"], marker="s", lw=2.0, color="#059669", label="val_cycle_ssim (A vs Rec A)")
ax.plot(df_val["epoch"], df_val["val_struct_ssim"], marker="^", lw=2.0, color="#0284C7", label="val_struct_ssim (A vs Fake B)")
ax.axhline(min_cycle_ssim, color="#DC2626", ls="--", lw=1.4, label=f"Cycle SSIM Floor ({min_cycle_ssim:.2f})")
ax.axvline(best_ep, color="#DC2626", ls=":", lw=1.2)
ax.set_ylim(0.68, 1.01)
ax.set_title("Anatomical Preservation (SSIM, Higher is Better)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("SSIM")
ax.legend(frameon=False, fontsize=8.2, loc="lower right")

# 4. Edit Magnitude vs Cycle Error
ax = axes[1, 1]
ax.plot(df_val["epoch"], df_val["val_delta_l1"], marker="o", lw=2.0, color="#D97706", label="val_delta_l1 |Fake B - Real A|")
ax.plot(df_val["epoch"], df_val["val_cycle_l1"], marker="d", lw=2.0, color="#475569", label="val_cycle_l1 |Rec A - Real A|")
ax.axhline(min_delta_l1, color="#DC2626", ls="--", lw=1.4, label=f"Collapse Guard Floor ({min_delta_l1:.3f})")
ax.axvline(best_ep, color="#DC2626", ls=":", lw=1.2)
ax.set_ylim(0.0, max(df_val["val_delta_l1"].max(), 0.14) * 1.15)
ax.set_title("Radiographic Edit Magnitude vs Cycle Error (L1)", fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean L1 Distance ([-1, 1] scale)")
ax.legend(frameon=False, fontsize=8.2, loc="upper right")

plt.tight_layout()
plt.show()


## 4. Qualitative Inspection: Validation Cycle Grids & Synthetic KL3 Pool
Inspects the **Best Checkpoint (Epoch 50)** bidirectional translations (`KL0 <-> KL3`), tracks how translation sharpens from Epoch 5 to Epoch 90 on the exact same validation joints, and visualizes signed pixel difference maps (`Syn KL3 - Real KL0`) across the 300-image synthetic pool.

In [ ]:
samples_dir = cg_out_dir / "samples"
best_grid_path = samples_dir / f"epoch_{best_ep:04d}.png"
if not best_grid_path.is_file():
    grids = sorted(samples_dir.glob("epoch_*.png"))
    best_grid_path = grids[-1]

grid_img = np.array(Image.open(best_grid_path).convert("L"))
H_g, W_g = grid_img.shape
# torchvision make_grid uses padding=2 around 256x256 tiles -> stride=258
stride = 258
n_cols = (W_g - 2) // stride
n_rows = (H_g - 2) // stride

row_names = [
    "Real KL0 (A)",
    "Syn KL3 (Fake B)",
    "Cycle KL0 (Rec A)",
    "Real KL3 (B)",
    "Syn KL0 (Fake A)",
    "Cycle KL3 (Rec B)",
][:n_rows]

fig, ax = plt.subplots(figsize=(10.5, 13.8), dpi=115)
ax.imshow(grid_img, cmap="gray", vmin=0, vmax=255)
ax.set_yticks([2 + r * stride + 128 for r in range(n_rows)])
ax.set_yticklabels(row_names, fontsize=10.5, fontweight="bold", color="#0F172A")
ax.set_xticks([2 + c * stride + 128 for c in range(n_cols)])
ax.set_xticklabels([f"Val Sample {c + 1}" for c in range(n_cols)], fontsize=10, fontweight="bold", color="#0F172A")
ax.xaxis.tick_top()
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title(f"Validation Bidirectional Translation Grid — Best Checkpoint ({best_grid_path.name})",
             fontsize=12.5, fontweight="bold", pad=22)
plt.tight_layout()
plt.show()

# Cross-epoch comparison on validation columns 0 and 1 (KL0 -> Fake KL3)
compare_epochs = [ep for ep in (5, 30, best_ep, int(df_loss["epoch"].max())) if (samples_dir / f"epoch_{ep:04d}.png").is_file()]
if compare_epochs:
    def _extract_tile(arr: np.ndarray, row: int, col: int, tile_size: int = 256, pad: int = 2) -> np.ndarray:
        y0 = pad + row * (tile_size + pad)
        x0 = pad + col * (tile_size + pad)
        return arr[y0 : y0 + tile_size, x0 : x0 + tile_size]

    first_arr = np.array(Image.open(samples_dir / f"epoch_{compare_epochs[0]:04d}.png").convert("L"))
    sample_cols = [0, 1] if n_cols >= 2 else [0]
    fig, axes = plt.subplots(len(sample_cols), len(compare_epochs) + 1, figsize=(14.5, 3.8 * len(sample_cols)), dpi=120)
    if len(sample_cols) == 1:
        axes = axes[np.newaxis, :]

    for r_idx, col_idx in enumerate(sample_cols):
        real_tile = _extract_tile(first_arr, row=0, col=col_idx)
        axes[r_idx, 0].imshow(real_tile, cmap="gray", vmin=0, vmax=255)
        axes[r_idx, 0].set_title(f"Real KL0 (Val #{col_idx + 1})", fontweight="bold", fontsize=9.5, pad=8)
        axes[r_idx, 0].axis("off")

        for c_idx, ep in enumerate(compare_epochs, start=1):
            ep_arr = np.array(Image.open(samples_dir / f"epoch_{ep:04d}.png").convert("L"))
            fake_tile = _extract_tile(ep_arr, row=1, col=col_idx)
            axes[r_idx, c_idx].imshow(fake_tile, cmap="gray", vmin=0, vmax=255)
            tag = " [BEST]" if ep == best_ep else ""
            title_color = "#DC2626" if ep == best_ep else "#0F172A"
            axes[r_idx, c_idx].set_title(f"Ep {ep} Syn KL3{tag}", fontweight="bold", fontsize=9.5, color=title_color, pad=8)
            axes[r_idx, c_idx].axis("off")

    fig.suptitle("Evolution of Synthesized KL3 Morphology Across Training Epochs (Same Validation KL0 Inputs)",
                 fontsize=12.0, fontweight="bold", y=0.97)
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.86, hspace=0.24, wspace=0.05)
    plt.show()


In [ ]:
syn_df = pd.read_csv(artifacts["synthetic_manifest.csv"])
assert not (syn_df["source_split"] == "test").any(), "Data leakage detected in synthetic manifest!"
assert not (syn_df["source_split"] == "val").any(), "Expected train-only synthetic source pool!"

syn_joined = syn_df.merge(
    manifest[["sample_id", "joint", "filename"]],
    left_on="source_sample_id",
    right_on="sample_id",
    how="left",
    suffixes=("", "_src"),
)

print(f"Synthetic KL3 pool size : {len(syn_joined)} images")
print(f"Source splits           : {syn_joined['source_split'].value_counts().to_dict()}")
print(f"Checkpoint epoch(s)     : {syn_joined['checkpoint_epoch'].value_counts().to_dict()}")

# 1 sample per DIP joint (DIP2..DIP5) for visual + difference heatmap inspection
picked_rows = []
for jname in ["DIP2", "DIP3", "DIP4", "DIP5"]:
    sub = syn_joined[syn_joined["joint"] == jname]
    if len(sub) > 0:
        picked_rows.append(sub.iloc[0])
picked_df = pd.DataFrame(picked_rows)

fig = plt.figure(figsize=(16.0, 9.2), dpi=120)
gs = fig.add_gridspec(3, len(picked_df) + 2, width_ratios=[1.15] + [1.0] * len(picked_df) + [0.06], wspace=0.14, hspace=0.28)

# Left column (spanning rows): Synthetic Pool Joint Distribution
ax_bar = fig.add_subplot(gs[:, 0])
j_counts = syn_joined["joint"].value_counts().reindex(["DIP2", "DIP3", "DIP4", "DIP5"], fill_value=0)
bars = ax_bar.bar(j_counts.index, j_counts.values, color=["#2563EB", "#0284C7", "#059669", "#7C3AED"], alpha=0.9)
for bar, val in zip(bars, j_counts.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val}", ha="center", va="bottom", fontweight="bold", fontsize=9.5)
ax_bar.set_ylim(0, j_counts.max() * 1.18)
ax_bar.set_title(f"Synthetic KL3 Pool\n(n={len(syn_joined)}, Train Only)", fontweight="bold", fontsize=10.5)
ax_bar.set_ylabel("Image Count")

img_size = int(cyclegan_cfg["data"]["image_size"])
for col_i, (_, row) in enumerate(picked_df.iterrows(), start=1):
    src_path = images_dir / row["filename"]
    syn_path = Path(row["image_path"])
    if not syn_path.is_file():
        syn_path = PROJECT_ROOT / "outputs" / "cyclegan" / "synthetic_kl3" / "images" / syn_path.name

    src_pil = Image.open(src_path).convert("L").resize((img_size, img_size), Image.Resampling.BICUBIC)
    syn_pil = Image.open(syn_path).convert("L")

    src_arr = np.asarray(src_pil, dtype=np.float32) / 127.5 - 1.0
    syn_arr = np.asarray(syn_pil, dtype=np.float32) / 127.5 - 1.0
    diff_arr = syn_arr - src_arr
    mae = float(np.abs(diff_arr).mean())

    ax0 = fig.add_subplot(gs[0, col_i])
    ax0.imshow(src_pil, cmap="gray", vmin=0, vmax=255)
    ax0.set_title(f"{row['joint']} ({row['source_sample_id']})\nReal KL0 Source", fontweight="bold", fontsize=9.0)
    ax0.axis("off")

    ax1 = fig.add_subplot(gs[1, col_i])
    ax1.imshow(syn_pil, cmap="gray", vmin=0, vmax=255)
    ax1.set_title(f"CycleGAN Syn KL3\n(Ep {int(row['checkpoint_epoch'])})", fontweight="bold", fontsize=9.0, color="#1D4ED8")
    ax1.axis("off")

    ax2 = fig.add_subplot(gs[2, col_i])
    im = ax2.imshow(diff_arr, cmap="RdBu_r", vmin=-0.8, vmax=0.8)
    ax2.set_title(f"Diff (Syn KL3 - Real KL0)\nMean |delta| = {mae:.3f}", fontsize=8.8)
    ax2.axis("off")

cax = fig.add_subplot(gs[2, -1])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Intensity Change ([-1, 1])", fontsize=8.8)
fig.suptitle("Generated KL3 Synthetic Pool & Pixel-Level Morphological Edit Maps", fontsize=12.8, fontweight="bold", y=0.98)
plt.show()


## 5. Downstream VGG19 Evaluation on Locked Test Set (`outputs/evaluation/`)
Compares all 4 training strategies fairly on the **untouched holdout `test` split (`1,958` images across `526` patients)** to measure both 5-class KL grading performance and Binary OA (`KL >= 3`) screening performance.

In [ ]:
strategy_specs = [
    ("eval_baseline_ce", "1. Baseline (CE)", "#64748B"),
    ("eval_cyclegan_sd_ce", "2. CycleGAN SD + CE", "#2563EB"),
    ("eval_baseline_os_focal", "3. OS + Focal", "#D97706"),
    ("eval_cyclegan_sd_os_focal", "4. CycleGAN SD + OS + Focal", "#059669"),
]

eval_records = []
eval_payloads = {}
for key, label, color in strategy_specs:
    payload = json.loads(artifacts[key].read_text())
    eval_payloads[key] = (label, color, payload)
    tm = payload["test_metrics"]
    pc = tm["per_class_f1"]
    eval_records.append({
        "Strategy": label,
        "Train Samples": payload["train_samples"],
        "Syn KL3": payload["synthetic_count"],
        "Accuracy": tm["accuracy"],
        "Macro F1": tm["macro_f1"],
        "KL0 F1": pc["KL0"],
        "KL1 F1": pc["KL1"],
        "KL2 F1": pc["KL2"],
        "KL3 F1": pc["KL3"],
        "KL4 F1": pc["KL4"],
        "OA Sens": tm["binary_oa_sensitivity"],
        "OA Prec": tm["binary_oa_precision"],
        "OA F1": tm["binary_oa_f1"],
        "OA ROC-AUC": tm["binary_oa_roc_auc"],
    })

eval_df = pd.DataFrame(eval_records)
display(eval_df.round(4))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18.2, 5.8), dpi=140)

# Panel A: Aggregate & Binary OA metrics
agg_keys = ["Accuracy", "Macro F1", "OA F1", "OA Sens", "OA Prec", "OA ROC-AUC"]
agg_labels = ["Accuracy", "Macro F1", "OA F1", "OA Sens", "OA Prec", "OA AUC"]
x1 = np.arange(len(agg_keys))
w1 = 0.20
legend_handles = []
for idx, (key, label, color) in enumerate(strategy_specs):
    vals = eval_df.loc[eval_df["Strategy"] == label, agg_keys].iloc[0].to_numpy()
    bars = ax1.bar(x1 + (idx - 1.5) * w1, vals, width=w1, label=label, color=color, alpha=0.92)
    legend_handles.append(bars[0])
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=6.6, rotation=90)

ax1.set_ylim(0.0, 1.18)
ax1.set_xticks(x1)
ax1.set_xticklabels(agg_labels, fontweight="bold", fontsize=8.8)
ax1.set_ylabel("Score")
ax1.set_title("Aggregate & Binary OA (KL >= 3) Metrics", fontweight="bold")

# Panel B: Per-Class F1
kl_cols = ["KL0 F1", "KL1 F1", "KL2 F1", "KL3 F1", "KL4 F1"]
kl_short = ["KL0", "KL1", "KL2", "KL3", "KL4"]
x2 = np.arange(len(kl_cols))
for idx, (key, label, color) in enumerate(strategy_specs):
    vals = eval_df.loc[eval_df["Strategy"] == label, kl_cols].iloc[0].to_numpy()
    bars = ax2.bar(x2 + (idx - 1.5) * w1, vals, width=w1, label=label, color=color, alpha=0.92)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=6.6, rotation=90)

ax2.set_ylim(0.0, 1.12)
ax2.set_xticks(x2)
ax2.set_xticklabels(kl_short, fontweight="bold", fontsize=9.5)
ax2.set_ylabel("F1 Score")
ax2.set_title("Per-Class F1 Comparison Across KL0–KL4", fontweight="bold")

# Shared legend for Panels A & B placed cleanly above both panels
fig.legend(
    handles=legend_handles,
    labels=[s[1] for s in strategy_specs],
    loc="upper center",
    bbox_to_anchor=(0.34, 0.935),
    ncol=4,
    frameon=False,
    fontsize=8.4,
)

# Panel C: Synthetic Data Ablation Deltas
delta_metrics = ["Macro F1", "KL1 F1", "KL2 F1", "KL3 F1", "KL4 F1", "OA F1"]
delta_labels = ["Macro F1", "KL1 F1", "KL2 F1", "KL3 F1", "KL4 F1", "OA F1"]
ce_base = eval_df.iloc[0][delta_metrics].to_numpy(dtype=float)
ce_syn = eval_df.iloc[1][delta_metrics].to_numpy(dtype=float)
os_base = eval_df.iloc[2][delta_metrics].to_numpy(dtype=float)
os_syn = eval_df.iloc[3][delta_metrics].to_numpy(dtype=float)

d_ce = ce_syn - ce_base
d_os = os_syn - os_base
x3 = np.arange(len(delta_metrics))
w3 = 0.34

b_ce = ax3.bar(x3 - w3 / 2 - 0.015, d_ce, width=w3, label="(SD + CE) - Baseline CE", color="#2563EB", alpha=0.88)
b_os = ax3.bar(x3 + w3 / 2 + 0.015, d_os, width=w3, label="(SD + OS + Focal) - (OS + Focal)", color="#059669", alpha=0.88)
ax3.axhline(0.0, color="#0F172A", lw=1.1)

for bar_group, frac in ((b_ce, 0.30), (b_os, 0.70)):
    for bar in bar_group:
        val = bar.get_height()
        offset = 0.004 if val >= 0 else -0.005
        va = "bottom" if val >= 0 else "top"
        x_pos = bar.get_x() + bar.get_width() * frac
        ax3.text(x_pos, val + offset, f"{val:+.3f}", ha="center", va=va, fontsize=6.8)

ymin = min(d_ce.min(), d_os.min()) - 0.032
ymax = max(d_ce.max(), d_os.max()) + 0.035
ax3.set_ylim(ymin, ymax)
ax3.set_xticks(x3)
ax3.set_xticklabels(delta_labels, fontweight="bold", fontsize=8.4)
ax3.set_ylabel("Delta Score (+ = Synthetic Helps)")
ax3.set_title("Ablation: Impact of Adding +235 Syn KL3 Images", fontweight="bold")
ax3.legend(frameon=False, fontsize=8.0, loc="upper right")

fig.suptitle("Downstream VGG19 Evaluation on Locked Test Set (n=1,958 images, 526 unseen patients)", fontsize=13.2, fontweight="bold", y=0.985)
plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.90])
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17.2, 4.4), dpi=140)
kl_ticks = ["KL0", "KL1", "KL2", "KL3", "KL4"]

for ax, (key, label, color) in zip(axes, strategy_specs):
    _, _, payload = eval_payloads[key]
    cm = np.asarray(payload["test_metrics"]["confusion_matrix"], dtype=int)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums > 0)

    ax.imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#CBD5E1")

    for r in range(5):
        for c in range(5):
            cnt = cm[r, c]
            pct = 100.0 * cm_norm[r, c]
            txt_color = "white" if cm_norm[r, c] > 0.52 else "#0F172A"
            ax.text(c, r, f"{cnt}\n({pct:.0f}%)", ha="center", va="center", fontsize=7.6, color=txt_color, fontweight="bold" if r == c else "normal")

    # Highlight Binary OA (KL >= 3) true & predicted quadrant
    oa_rect = mpatches.Rectangle((2.5, 2.5), 2.0, 2.0, fill=False, edgecolor="#DC2626", lw=2.0, ls="--")
    ax.add_patch(oa_rect)

    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(kl_ticks, fontsize=8.8)
    ax.set_yticklabels(kl_ticks if ax is axes[0] else [], fontsize=8.8)
    ax.set_xlabel("Predicted KL", fontsize=9.0)
    if ax is axes[0]:
        ax.set_ylabel("True KL", fontsize=9.0)
    tm = payload["test_metrics"]
    ax.set_title(f"{label}\nMacro F1={tm['macro_f1']:.3f} | OA F1={tm['binary_oa_f1']:.3f}", fontweight="bold", fontsize=9.4, color=color if color != "#64748B" else "#0F172A")

fig.suptitle("Locked Test Set Confusion Matrices (Row-Normalized Recall % & Raw Counts; Red Box = KL>=3 OA Quadrant)",
             fontsize=12.2, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()


## 6. Full CycleGAN Training (Optional Execution)

- **Config (`configs/cyclegan.yaml`)**: `200` epochs, `batch_size=16`, `num_workers=4`, `lr=0.0002`, AMP `bfloat16`
- **Early Stopping**: `enabled=True`, `eval_every_epochs=5`, `min_epochs=40`, `patience=8` checks
- **Safety Guard**: `RUN_TRAINING = False` so plotting runs never overwrite existing checkpoints
- **Interactive Run**: Set `RUN_TRAINING = True` (and optionally `RESUME_CHECKPOINT`) on a GPU node
- **BU SCC 2-GPU DDP Batch Job**:
```bash
qsub jobs/train_cyclegan.qsub
```

In [ ]:
from hand_oa.utils import merge_configs
from cyclegan.engine import CycleGANTrainer

RUN_TRAINING = False        # Set True on a GPU node to run/resume full training

# Full YAML defaults (override only if intentionally testing a subset)
MAX_EPOCHS = None           # None -> 200 epochs from configs/cyclegan.yaml
MAX_STEPS_PER_EPOCH = None  # None -> iterate full epoch
RESUME_CHECKPOINT = None    # e.g., "outputs/cyclegan/checkpoints/epoch_0090.pt"
BATCH_SIZE_OVERRIDE = None  # None -> batch_size=16 from configs/cyclegan.yaml
OUTPUT_DIR_OVERRIDE = None  # None -> "outputs/cyclegan"

train_overrides = {}
if BATCH_SIZE_OVERRIDE is not None:
    train_overrides.setdefault("data", {})["batch_size"] = BATCH_SIZE_OVERRIDE
if OUTPUT_DIR_OVERRIDE is not None:
    train_overrides.setdefault("experiment", {})["output_dir"] = OUTPUT_DIR_OVERRIDE
train_cfg = merge_configs(cyclegan_cfg, train_overrides) if train_overrides else cyclegan_cfg

print(
    "Full Training Config -> "
    f"epochs={MAX_EPOCHS or train_cfg['training']['epochs']}, "
    f"batch_size={train_cfg['data']['batch_size']}, "
    f"num_workers={train_cfg['data']['num_workers']}, "
    f"lr={train_cfg['training']['lr']}, "
    f"use_amp={train_cfg['training']['use_amp']}, "
    f"early_stopping={train_cfg['early_stopping']['enabled']} "
    f"(patience={train_cfg['early_stopping']['patience']} checks), "
    f"output_dir={train_cfg['experiment']['output_dir']}"
)

if RUN_TRAINING:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        print("WARNING: Running on CPU. Use a GPU node or `qsub jobs/train_cyclegan.qsub` for full 200-epoch training.")
    trainer = CycleGANTrainer(
        data_cfg=data_cfg,
        cyclegan_cfg=train_cfg,
        project_root=PROJECT_ROOT,
        device=device,
    )
    history = trainer.fit(
        max_epochs=MAX_EPOCHS,
        max_steps_per_epoch=MAX_STEPS_PER_EPOCH,
        resume_checkpoint=RESUME_CHECKPOINT,
    )
    print(f"Training completed up to epoch {history[-1]['epoch']}. Outputs saved to: {trainer.output_dir}")
else:
    print("RUN_TRAINING=False -> Training skipped to preserve existing checkpoints.")


## 7. Full Synthetic KL3 Generation & BU SCC Batch Commands

- **Config (`configs/cyclegan.yaml`)**: `source_split="train"` (`test` strictly blocked), `direction="a2b"` (`KL0 -> KL3`)
- **Output Pool**: `max_samples=300` images generated from `best_checkpoint.pt` into `outputs/cyclegan/synthetic_kl3/`
- **Safety Guard**: `RUN_SYNTHESIS = False` so plotting runs preserve existing synthetic images
- **Interactive Run**: Set `RUN_SYNTHESIS = True` in the cell below to regenerate the pool
- **Downstream VGG19 Evaluation on Locked `test` Set**:
```bash
qsub jobs/eval_cyclegan.qsub
```

In [ ]:
from hand_oa.utils import merge_configs
from cyclegan.engine import generate_synthetic_images

RUN_SYNTHESIS = False           # Set True to generate the full KL3 synthetic pool

# Full YAML defaults (test split is strictly blocked inside generate_synthetic_images)
CHECKPOINT_PATH = artifacts["best_checkpoint.pt"]
SOURCE_SPLIT_OVERRIDE = None    # None -> "train" from configs/cyclegan.yaml
MAX_SAMPLES_OVERRIDE = None     # None -> 300 from configs/cyclegan.yaml
SYN_OUTPUT_DIR_OVERRIDE = None  # None -> "outputs/cyclegan/synthetic_kl3"

gen_overrides = {}
if SOURCE_SPLIT_OVERRIDE is not None:
    gen_overrides.setdefault("generation", {})["source_split"] = SOURCE_SPLIT_OVERRIDE
if MAX_SAMPLES_OVERRIDE is not None:
    gen_overrides.setdefault("generation", {})["max_samples"] = MAX_SAMPLES_OVERRIDE
gen_cfg = merge_configs(cyclegan_cfg, gen_overrides) if gen_overrides else cyclegan_cfg

print(
    "Full Synthesis Config -> "
    f"checkpoint={CHECKPOINT_PATH.name}, "
    f"source_split={gen_cfg['generation']['source_split']}, "
    f"direction={gen_cfg['generation']['direction']} "
    f"(KL{gen_cfg['data']['domain_a_grades']} -> KL{gen_cfg['generation']['target_label']}), "
    f"max_samples={gen_cfg['generation']['max_samples']}, "
    f"output_subdir={gen_cfg['generation']['output_subdir']}"
)

if RUN_SYNTHESIS:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    syn_manifest_out = generate_synthetic_images(
        data_cfg=data_cfg,
        cyclegan_cfg=gen_cfg,
        checkpoint_path=CHECKPOINT_PATH,
        project_root=PROJECT_ROOT,
        output_dir=SYN_OUTPUT_DIR_OVERRIDE,
        device=device,
    )
    syn_preview = pd.read_csv(syn_manifest_out)
    print(f"Generated {len(syn_preview)} synthetic images -> {syn_manifest_out}")
    print("Source splits:", syn_preview["source_split"].value_counts().to_dict())
else:
    print("RUN_SYNTHESIS=False -> Synthesis skipped to preserve existing outputs/cyclegan/synthetic_kl3.")
